# Semana 10 | Práctico 2.5: Enfoques de alineamiento de secuencias
**Bioinformática en el Diagnóstico Molecular**

### Resultados esperados
Al finalizar podrás:
- interpretar *matches*, *mismatches*, gaps y score;
- comparar alineamiento global y local;
- aplicar matrices de sustitución a ADN y proteínas;
- interpretar un dotplot;
- reconocer el propósito de un alineamiento múltiple (MSA).

**Archivos usados:** `MyBRCA.fasta`, `myFasta.fasta` y `fastaMSA.fasta`.

> Antes de comenzar, sube los tres FASTA al panel **Archivos** de Colab.


## 1. Preparación del entorno


In [ ]:
if (!requireNamespace("BiocManager", quietly=TRUE))
    install.packages("BiocManager", repos="https://cloud.r-project.org")

for (p in c("Biostrings","pwalign","msa")) {
    if (!requireNamespace(p, quietly=TRUE))
        BiocManager::install(p, ask=FALSE, update=FALSE)
}
if (!requireNamespace("seqinr", quietly=TRUE))
    install.packages("seqinr", repos="https://cloud.r-project.org")

library(Biostrings)
library(pwalign)
library(msa)
library(seqinr)


## 2. ¿Cómo se construye un alineamiento?

Primero trabajaremos con dos secuencias pequeñas para poder **ver** lo que hace el algoritmo.

Antes de ejecutar:
1. ¿Qué posiciones parecen coincidir?
2. ¿Dónde esperas encontrar diferencias?
3. ¿Crees que será necesario introducir un gap?


In [ ]:
seq1 <- DNAString("GAATTCGGCTA")
seq2 <- DNAString("GATTACCTA")

seq1
seq2

matriz_ADN <- nucleotideSubstitutionMatrix(
    match=1, mismatch=-1, baseOnly=TRUE
)


### 2.1 Alineamiento global


In [ ]:
aln_global_corto <- pairwiseAlignment(
    seq1, seq2,
    type="global",
    substitutionMatrix=matriz_ADN,
    gapOpening=2,
    gapExtension=1
)

aln_global_corto
score(aln_global_corto)


**Discusión guiada**
- Identifica matches, mismatches y gaps.
- ¿Qué representa el score?
- Si cambiáramos las penalizaciones de gap, ¿podría cambiar el alineamiento óptimo?


### 2.2 Las mismas secuencias, ahora con alineamiento local


In [ ]:
aln_local_corto <- pairwiseAlignment(
    seq1, seq2,
    type="local",
    substitutionMatrix=matriz_ADN,
    gapOpening=2,
    gapExtension=1
)

aln_local_corto
score(aln_local_corto)


**Pregunta clave:** ¿por qué global y local pueden producir resultados diferentes si las secuencias de entrada son exactamente las mismas?


# 3. Caso real: BRCA1 y elección global/local

Cargaremos `MyBRCA.fasta`. Antes de alinear, inspeccionaremos cuántas secuencias contiene y sus longitudes.


In [ ]:
brca <- readDNAStringSet("MyBRCA.fasta")
brca
names(brca)
width(brca)


### Antes de ejecutar
Observa las longitudes.

1. ¿Son comparables de extremo a extremo?
2. Si una secuencia corresponde a una región corta y la otra es mucho más extensa, ¿qué problema podría tener un alineamiento global?
3. **Predice:** ¿global o local debería ser más informativo?


In [ ]:
brca1 <- brca[[1]]
brca2 <- brca[[2]]

aln_brca_global <- pairwiseAlignment(
    brca1, brca2,
    type="global",
    substitutionMatrix=matriz_ADN,
    gapOpening=5,
    gapExtension=2
)

aln_brca_local <- pairwiseAlignment(
    brca1, brca2,
    type="local",
    substitutionMatrix=matriz_ADN,
    gapOpening=5,
    gapExtension=2
)

aln_brca_global
score(aln_brca_global)

aln_brca_local
score(aln_brca_local)


### Interpretación BRCA1
- ¿Qué parte de las secuencias intenta utilizar el global?
- ¿Qué región conserva el local?
- ¿Cuál responde mejor a la pregunta “¿dónde se encuentra la región corta dentro de la secuencia extensa?”?
- ¿Por qué **no debemos comparar scores de forma aislada** sin considerar el tipo de alineamiento y los parámetros?


# 4. Proteínas reales: hemoglobina beta (HBB)

`myFasta.fasta` contiene HBB de humano, chimpancé, gallina, pez cebra y caballo.

Primero inspeccionaremos el archivo.


In [ ]:
hbb <- readAAStringSet("myFasta.fasta")
hbb
names(hbb)
width(hbb)


## 4.1 Humano vs. chimpancé

Antes de alinear: ¿esperas muchas o pocas diferencias?


In [ ]:
data(BLOSUM62)

hbb_humano <- hbb[grep("HUMAN", names(hbb))][[1]]
hbb_chimp  <- hbb[grep("PANTR", names(hbb))][[1]]
hbb_gallina <- hbb[grep("CHICK", names(hbb))][[1]]

aln_hum_chimp <- pairwiseAlignment(
    hbb_humano, hbb_chimp,
    type="global",
    substitutionMatrix=BLOSUM62,
    gapOpening=10,
    gapExtension=0.5
)

aln_hum_chimp
score(aln_hum_chimp)


## 4.2 Humano vs. gallina

Usaremos exactamente los mismos parámetros.


In [ ]:
aln_hum_gallina <- pairwiseAlignment(
    hbb_humano, hbb_gallina,
    type="global",
    substitutionMatrix=BLOSUM62,
    gapOpening=10,
    gapExtension=0.5
)

aln_hum_gallina
score(aln_hum_gallina)


### Comparación
1. ¿Qué par presenta mayor similitud?
2. ¿Qué observas en los scores?
3. ¿Por qué BLOSUM62 es más informativa para proteínas que asignar simplemente +1 a igualdad y −1 a cualquier sustitución?
4. ¿Un aminoácido diferente implica necesariamente una sustitución con el mismo “costo” que cualquier otra?


# 5. Dotplot: otra forma de visualizar similitud

Un dotplot permite observar patrones de similitud entre dos secuencias. Una diagonal continua sugiere similitud a lo largo de gran parte de ambas secuencias.

Compararemos humano–chimpancé y humano–gallina.


In [ ]:
par(mfrow=c(1,1))
dotPlot(
    s2c(as.character(hbb_humano)),
    s2c(as.character(hbb_chimp)),
    main="HBB: Humano vs. chimpancé"
)

In [ ]:
dotPlot(
    s2c(as.character(hbb_humano)),
    s2c(as.character(hbb_gallina)),
    main="HBB: Humano vs. gallina"
)

### Interpreta los dotplots
- ¿En cuál observas una diagonal más continua?
- ¿Qué significan las interrupciones?
- ¿Coincide la interpretación visual con lo observado en los alineamientos?


# 6. Alineamiento múltiple: COX2

Hasta ahora comparamos dos secuencias. ¿Qué ocurre cuando queremos comparar varias especies simultáneamente?

`fastaMSA.fasta` contiene secuencias de COX2 de distintas especies.


In [ ]:
cox2 <- readAAStringSet("fastaMSA.fasta")
cox2
names(cox2)
width(cox2)


### Predicción
Antes del MSA:
- ¿esperas que todas las posiciones sean iguales?
- ¿qué tipo de información puede entregar un MSA que no entrega una comparación de sólo dos secuencias?


In [ ]:
alineamiento_msa <- msa(
    cox2,
    method="ClustalW",
    order="input"
)

alineamiento_msa


### Interpretación del MSA
Observa el alineamiento y discute:

1. ¿Existen regiones conservadas entre varias especies?
2. ¿Dónde se observan posiciones variables?
3. ¿Qué secuencia parece más divergente respecto de las demás?
4. ¿Qué podría sugerir una región muy conservada acerca de su importancia biológica?

> El objetivo aquí no es memorizar ClustalW, sino comprender **qué pregunta permite responder un alineamiento múltiple**.


# 7. Cierre: elige la estrategia

Indica qué utilizarías y justifica:

**A.** Comparar dos versiones completas del mismo gen de longitud similar.  
**B.** Buscar un fragmento de 150 pb dentro de una referencia de 5.000 pb.  
**C.** Identificar posiciones conservadas de una proteína entre seis especies.

Opciones: **global / local / múltiple**.

### Idea final
**No existe un alineamiento universalmente “mejor”: la pregunta biológica determina la estrategia.**
